# 🔬 Reconstruct v6 Engineered (REAL DATA ONLY)

**CRITICAL:** Zero synthetic data. Every feature from real source.

**Strategy:**
1. Keep real `id`, `date`, `gender`, `insulin_type_bolus`
2. Calculate REAL temporal features from `date`
3. Calculate REAL trends per patient using `id`
4. Apply toxic cleaning & balancing

**Output:**
- `master_train_v6_real.parquet` (balanced)
- `master_test_v6_real.parquet` (natural distribution)

## 1. Setup

In [ ]:
!pip install polars -q

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import polars as pl
import numpy as np
import os

print(f"Polars: {pl.__version__}")

In [ ]:
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"
RAW_PATH = f"{BASE_PATH}/raw"
OUTPUT_PATH = f"{BASE_PATH}/processed"

FILES = {
    "train_raw": f"{RAW_PATH}/metabonet_public_train.parquet",
    "test_raw": f"{RAW_PATH}/metabonet_public_test.parquet",
    "train_out": f"{OUTPUT_PATH}/master_train_v6_real.parquet",
    "test_out": f"{OUTPUT_PATH}/master_test_v6_real.parquet"
}

for name, path in FILES.items():
    if "_raw" in name:
        exists = "✅" if os.path.exists(path) else "❌"
        print(f"{exists} {name}: {path}")

## 2. Data Selection (Source of Truth)

In [ ]:
# CRITICAL: Must include id, date for real features
REQUIRED_COLS = [
    'id', 'date', 'CGM', 'bolus', 'carbs', 'basal', 
    'age', 'weight', 'gender', 'insulin_type_bolus'
]

print(f"📋 Required columns: {REQUIRED_COLS}")

In [ ]:
def scan_raw(path, label):
    """Scan raw parquet and select required columns."""
    print(f"\n📂 Scanning {label}...")
    
    lf = pl.scan_parquet(path)
    all_cols = lf.columns
    
    # Check which required columns exist
    available = [c for c in REQUIRED_COLS if c in all_cols]
    missing = [c for c in REQUIRED_COLS if c not in all_cols]
    
    print(f"   ✅ Available: {available}")
    if missing:
        print(f"   ❌ Missing: {missing}")
    
    return lf.select(available), available

lf_train, train_cols = scan_raw(FILES["train_raw"], "TRAIN")
lf_test, test_cols = scan_raw(FILES["test_raw"], "TEST")

In [ ]:
# Fill nulls in carbs and basal
def fill_nulls(lf):
    return lf.with_columns([
        pl.col("carbs").fill_null(0.0),
        pl.col("basal").fill_null(0.0)
    ])

lf_train = fill_nulls(lf_train)
lf_test = fill_nulls(lf_test)
print("✅ Nulls filled in carbs and basal")

## 3. Toxic Cleaning

In [ ]:
def get_toxic_mask():
    """DROP rows where low CGM, no carbs, but bolus given."""
    return (
        (pl.col("CGM") < 110) & 
        (pl.col("carbs") == 0) & 
        (pl.col("bolus") > 0)
    )

lf_train_clean = lf_train.filter(~get_toxic_mask())
lf_test_clean = lf_test.filter(~get_toxic_mask())

print("✅ Toxic mask applied")

## 4. Collect & Balance Training Data

In [ ]:
print("📊 Collecting training data (streaming)...")
df_train = lf_train_clean.collect(streaming=True)
print(f"   Rows: {len(df_train):,}")

In [ ]:
# Split into action and safe pools
print("\n⚖️ Balancing training data...")

pool_action = df_train.filter(pl.col("bolus") > 0)
pool_safe = df_train.filter(pl.col("bolus") == 0)

n_action = len(pool_action)
n_safe = len(pool_safe)

print(f"   Action (bolus>0): {n_action:,}")
print(f"   Safe (bolus=0): {n_safe:,}")

In [ ]:
# Sample safe to match action
target_safe = min(n_action, n_safe)

# Balance by CGM range (50% low, 50% high)
pool_safe_low = pool_safe.filter(pl.col("CGM") < 120)
pool_safe_high = pool_safe.filter(pl.col("CGM") >= 120)

sample_low = min(target_safe // 2, len(pool_safe_low))
sample_high = min(target_safe - sample_low, len(pool_safe_high))

print(f"   Sampling: {sample_low:,} low + {sample_high:,} high")

if sample_low > 0:
    pool_safe_low = pool_safe_low.sample(n=sample_low, seed=42)
if sample_high > 0:
    pool_safe_high = pool_safe_high.sample(n=sample_high, seed=42)

pool_safe_balanced = pl.concat([pool_safe_low, pool_safe_high])

In [ ]:
# Merge and shuffle
df_train_balanced = pl.concat([pool_action, pool_safe_balanced])
df_train_balanced = df_train_balanced.sample(fraction=1.0, seed=42, shuffle=True)

print(f"\n✅ Balanced train: {len(df_train_balanced):,} rows")

## 5. Collect Test Data (No Balancing)

In [ ]:
print("📊 Collecting test data (streaming)...")
df_test = lf_test_clean.collect(streaming=True)
print(f"   Rows: {len(df_test):,}")

## 6. REAL Feature Engineering

In [ ]:
def add_real_temporal_features(df):
    """Add temporal features from REAL date column."""
    print("   ⏰ Adding REAL temporal features...")
    
    if 'date' not in df.columns:
        print("   ⚠️ 'date' column missing - cannot calculate real temporal features")
        return df
    
    # Ensure date is datetime
    if df['date'].dtype != pl.Datetime:
        df = df.with_columns(pl.col('date').cast(pl.Datetime))
    
    df = df.with_columns([
        # Extract hour and calculate sin/cos
        (2 * np.pi * pl.col('date').dt.hour() / 24).sin().alias('time_sin'),
        (2 * np.pi * pl.col('date').dt.hour() / 24).cos().alias('time_cos'),
        # Day of week (0=Monday, 6=Sunday)
        pl.col('date').dt.weekday().alias('day_of_week'),
        # Is weekend
        (pl.col('date').dt.weekday() >= 5).cast(pl.Int8).alias('is_weekend')
    ])
    
    return df

print("✅ add_real_temporal_features defined")

In [ ]:
def add_real_trend_features(df):
    """Add CGM trend features using REAL patient ID for grouping."""
    print("   📈 Adding REAL trend features (per patient)...")
    
    if 'id' not in df.columns:
        print("   ⚠️ 'id' column missing - using global diff")
        df = df.with_columns([
            pl.col('CGM').diff().fill_null(0).alias('cgm_delta_5min'),
            pl.col('CGM').diff(n=3).fill_null(0).alias('cgm_delta_15min')
        ])
    else:
        # REAL per-patient trends
        df = df.sort(['id', 'date'])  # Sort by patient and time
        df = df.with_columns([
            pl.col('CGM').diff().over('id').fill_null(0).alias('cgm_delta_5min'),
            pl.col('CGM').diff(n=3).over('id').fill_null(0).alias('cgm_delta_15min')
        ])
    
    return df

print("✅ add_real_trend_features defined")

In [ ]:
def add_real_categorical_encodings(df):
    """Encode REAL gender and insulin_type."""
    print("   🏷️ Adding REAL categorical encodings...")
    
    # Gender encoding
    if 'gender' in df.columns:
        df = df.with_columns(
            pl.when(pl.col('gender') == 'Male').then(0)
            .when(pl.col('gender') == 'Female').then(1)
            .otherwise(0)
            .cast(pl.Float32)
            .alias('gender_encoded')
        )
    else:
        df = df.with_columns(pl.lit(0.0).alias('gender_encoded'))
    
    # Insulin type encoding
    if 'insulin_type_bolus' in df.columns:
        df = df.with_columns(
            pl.when(pl.col('insulin_type_bolus') == 'Humalog').then(0)
            .when(pl.col('insulin_type_bolus') == 'NovoRapid').then(1)
            .when(pl.col('insulin_type_bolus') == 'Apidra').then(2)
            .when(pl.col('insulin_type_bolus') == 'Fiasp').then(3)
            .when(pl.col('insulin_type_bolus') == 'Lyumjev').then(4)
            .otherwise(0)
            .cast(pl.Float32)
            .alias('insulin_type_encoded')
        )
    else:
        df = df.with_columns(pl.lit(0.0).alias('insulin_type_encoded'))
    
    return df

print("✅ add_real_categorical_encodings defined")

In [ ]:
def apply_real_pipeline(df, name):
    """Apply full REAL feature engineering pipeline."""
    print(f"\n🔧 Processing {name} with REAL features...")
    print(f"   Input: {len(df):,} rows, {len(df.columns)} cols")
    
    df = add_real_temporal_features(df)
    df = add_real_trend_features(df)
    df = add_real_categorical_encodings(df)
    
    # Cast to Float32
    numeric_cols = [c for c in df.columns if df[c].dtype in [pl.Float64, pl.Int64, pl.Int32]]
    df = df.cast({c: pl.Float32 for c in numeric_cols if c != 'id'})
    
    print(f"   Output: {len(df):,} rows, {len(df.columns)} cols")
    return df

print("✅ apply_real_pipeline defined")

In [ ]:
# Apply pipeline
df_train_final = apply_real_pipeline(df_train_balanced, "TRAIN")
df_test_final = apply_real_pipeline(df_test, "TEST")

## 7. Export

In [ ]:
print(f"\n💾 Saving train: {FILES['train_out']}")
df_train_final.write_parquet(FILES['train_out'])
print("✅ Train saved!")

print(f"\n💾 Saving test: {FILES['test_out']}")
df_test_final.write_parquet(FILES['test_out'])
print("✅ Test saved!")

## 8. Verification

In [ ]:
print("\n" + "=" * 60)
print("🔍 VERIFICATION - REAL DATA CHECK")
print("=" * 60)

print(f"\n📋 TRAIN columns ({len(df_train_final.columns)}):")
print(df_train_final.columns)

print(f"\n📋 TEST columns ({len(df_test_final.columns)}):")
print(df_test_final.columns)

In [ ]:
# Check for real features
print("\n🔍 REAL DATA VALIDATION:")

# Check time_sin range (should be -1 to 1)
if 'time_sin' in df_train_final.columns:
    t_min = df_train_final['time_sin'].min()
    t_max = df_train_final['time_sin'].max()
    print(f"   time_sin range: [{t_min:.3f}, {t_max:.3f}] (expected: [-1, 1])")

# Check cgm_delta variance (should not be 0 if real)
if 'cgm_delta_5min' in df_train_final.columns:
    delta_std = df_train_final['cgm_delta_5min'].std()
    print(f"   cgm_delta_5min std: {delta_std:.3f} (should be > 0 if real)")

# Check if id column preserved
if 'id' in df_train_final.columns:
    n_patients = df_train_final['id'].n_unique()
    print(f"   Unique patients (id): {n_patients:,}")
    print("   ✅ Real patient IDs preserved!")
else:
    print("   ⚠️ 'id' column missing - patient grouping lost")

In [ ]:
print("\n" + "=" * 60)
print("📋 FINAL SUMMARY")
print("=" * 60)

train_action = len(df_train_final.filter(pl.col('bolus') > 0))
train_safe = len(df_train_final.filter(pl.col('bolus') == 0))

test_action = len(df_test_final.filter(pl.col('bolus') > 0))
test_safe = len(df_test_final.filter(pl.col('bolus') == 0))

print(f"""
TRAIN (Balanced):
   Total: {len(df_train_final):,}
   Action: {train_action:,} ({train_action/len(df_train_final)*100:.1f}%)
   Safe: {train_safe:,} ({train_safe/len(df_train_final)*100:.1f}%)
   Columns: {len(df_train_final.columns)}

TEST (Natural Distribution):
   Total: {len(df_test_final):,}
   Action: {test_action:,} ({test_action/len(df_test_final)*100:.1f}%)
   Safe: {test_safe:,} ({test_safe/len(df_test_final)*100:.1f}%)
   Columns: {len(df_test_final.columns)}

FILES:
   ✅ {FILES['train_out']}
   ✅ {FILES['test_out']}

🎯 ZERO SYNTHETIC DATA - ALL FEATURES FROM REAL SOURCE
""")
print("=" * 60)
print("🎉 DONE!")